# 🐾 Pet Breed Classification: Starter Notebook

Predict the breed (37 classes, labels `0`-`36`) of a pet from its photo. Build the best model you can and submit `submission.csv`.

**Provided:** setup, the dataframes (`train_df`, `test_df`, `sample`) and the final submission cells (section 3).
**Your job:** everything in section 2: build the Dataset and DataLoaders yourself, then the model, training and tuning.

**Kaggle checklist:** attach the competition data in **Input** · enable a **GPU** in Settings → Accelerator · run **Save Version → Save & Run All** · open the version's **Output** tab · **Submit to Competition**.

**Contract with section 3:**
- `model`: your trained network, with its best weights saved as `best.pt` (a `state_dict`).
- `ts_dl`: a test `DataLoader` with `shuffle=False` that yields **image tensors only** (no labels), in the same order as `test_df['id']`.

## 1. Setup and data files (provided)

In [28]:
import os, random, time
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.optim as op
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from PIL import Image
from sklearn.model_selection import train_test_split

CFG = dict(img_size=128, batch_size=32, epochs=15, num_workers=2, seed=42, lr=1e-3, val_frac=0.15)

def seed_everything(s: int) -> None:
    random.seed(s); np.random.seed(s)
    torch.manual_seed(s); torch.cuda.manual_seed_all(s)

seed_everything(CFG["seed"])
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

ROOT = next(Path('/kaggle/input').rglob("train.csv")).parent
train_df = pd.read_csv(ROOT / 'train.csv')
test_df  = pd.DataFrame({"id": sorted(p.stem for p in (ROOT / 'test').glob('*.jpg'))})
sample   = pd.read_csv(ROOT / "sample_submission.csv")
print(device, "|", train_df.shape, "|", train_df['label'].nunique(), "classes")

cuda | (5144, 3) | 37 classes


In [29]:
print("Training data:")
print(train_df.head())

print("\nShape:", train_df.shape)

print("\nColumns:", train_df.columns.tolist())

print("\nNumber of breeds:", train_df['label'].nunique())

print("\nImages per breed:")
print(train_df['label'].value_counts().sort_index())

Training data:
          id  label        label_name
0  img_06102     16          havanese
1  img_01882     16          havanese
2  img_06405      6            Birman
3  img_00375     30  scottish_terrier
4  img_02107      5            Bengal

Shape: (5144, 3)

Columns: ['id', 'label', 'label_name']

Number of breeds: 37

Images per breed:
label
0     139
1     140
2     140
3     140
4     140
5     140
6     140
7     129
8     139
9     140
10    140
11    133
12    137
13    140
14    140
15    140
16    140
17    140
18    139
19    140
20    140
21    140
22    137
23    140
24    140
25    140
26    140
27    140
28    140
29    140
30    139
31    140
32    139
33    140
34    133
35    140
36    140
Name: count, dtype: int64


## 2. Your work

Images are in `ROOT/'train'` and `ROOT/'test'` as `<id>.jpg`, labels are in `train_df`. Build everything here (add as many cells as you need):

`Dataset` → `DataLoader`s (train / validation / test) → model → training → tuning.

Requirements for section 3: `model`, `best.pt` (`torch.save(model.state_dict(), 'best.pt')`) and `ts_dl` (see the top of the notebook).

In [30]:
# your Dataset, DataLoaders, model, training loop, experiments ...
transform = T.Compose([
    T.Resize((CFG["img_size"], CFG["img_size"])),
    T.ToTensor()
])

In [31]:
class PetDataset(Dataset):

    def __init__(self, dataframe, image_dir, transform=None):
        self.df = dataframe.reset_index(drop=True)
        self.image_dir = image_dir
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, index):

        row = self.df.iloc[index]

        image_id = row["id"]
        label = row["label"]

        image_path = self.image_dir / f"{image_id}.jpg"

        image = Image.open(image_path).convert("RGB")

        if self.transform:
            image = self.transform(image)

        return image, torch.tensor(label, dtype=torch.long)

In [32]:
train_part, val_part = train_test_split(
    train_df,
    test_size=CFG["val_frac"],
    random_state=123,
    stratify=train_df["label"]
)

In [33]:
train_ds = PetDataset(
    train_part,
    ROOT / "train",
    transform
)

val_ds = PetDataset(
    val_part,
    ROOT / "train",
    transform
)

In [34]:
image , label = train_ds[0]
print(image)

tensor([[[0.3020, 0.3216, 0.3333,  ..., 0.1020, 0.1569, 0.2588],
         [0.1412, 0.2431, 0.2863,  ..., 0.1059, 0.2510, 0.3804],
         [0.0745, 0.2157, 0.3098,  ..., 0.1843, 0.3098, 0.3922],
         ...,
         [0.8157, 0.8157, 0.8118,  ..., 0.3765, 0.3255, 0.1922],
         [0.8118, 0.8157, 0.8118,  ..., 0.4314, 0.4784, 0.4078],
         [0.8118, 0.8196, 0.8000,  ..., 0.4157, 0.4941, 0.5059]],

        [[0.2588, 0.2784, 0.2784,  ..., 0.0745, 0.1216, 0.2039],
         [0.1451, 0.2275, 0.2471,  ..., 0.0902, 0.2118, 0.2980],
         [0.0706, 0.1804, 0.2510,  ..., 0.1686, 0.2549, 0.2941],
         ...,
         [0.8588, 0.8588, 0.8549,  ..., 0.2980, 0.2588, 0.1529],
         [0.8549, 0.8627, 0.8549,  ..., 0.3412, 0.3961, 0.3490],
         [0.8549, 0.8627, 0.8510,  ..., 0.3490, 0.4235, 0.4353]],

        [[0.0980, 0.1216, 0.1529,  ..., 0.0039, 0.0353, 0.1098],
         [0.0275, 0.0706, 0.1255,  ..., 0.0118, 0.1059, 0.2039],
         [0.0157, 0.0627, 0.1412,  ..., 0.0667, 0.1529, 0.

In [35]:
train_dl = DataLoader(
    train_ds,
    batch_size=CFG["batch_size"],
    shuffle=True,
    num_workers=CFG["num_workers"]
)
val_dl = DataLoader(
    val_ds,
    batch_size=CFG["batch_size"],
    shuffle=False,
    num_workers=CFG["num_workers"]
)

In [36]:
images, labels = next(iter(train_dl))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 128, 128])
torch.Size([32])


In [37]:
class PetCNN(nn.Module):

    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(

            # First convolutional block
            nn.Conv2d(3, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Second convolutional block
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),

            # Third convolutional block
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2)
        )

        self.classifier = nn.Sequential(

            nn.Flatten(),

            nn.Linear(128 * 16 * 16, 256),
            nn.ReLU(),

            nn.Dropout(0.2),
            
            nn.Linear(256, 37)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x

In [38]:
model = PetCNN().to(device)

print(model)
criterion = nn.CrossEntropyLoss()

optimizer = op.Adam(
    model.parameters(),
    lr=CFG["lr"]
)

PetCNN(
  (features): Sequential(
    (0): Conv2d(3, 32, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU()
    (2): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (3): Conv2d(32, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (4): ReLU()
    (5): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (6): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (7): ReLU()
    (8): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
  )
  (classifier): Sequential(
    (0): Flatten(start_dim=1, end_dim=-1)
    (1): Linear(in_features=32768, out_features=256, bias=True)
    (2): ReLU()
    (3): Dropout(p=0.2, inplace=False)
    (4): Linear(in_features=256, out_features=37, bias=True)
  )
)


In [39]:
def train_one_epoch(model, train_dl, criterion, optimizer, device):

    model.train()

    total_loss = 0
    correct = 0
    total = 0

    for images, labels in train_dl:

        images = images.to(device)
        labels = labels.to(device)

        # Forward pass
        outputs = model(images)

        # Calculate loss
        loss = criterion(outputs, labels)

        # Backpropagation
        optimizer.zero_grad()
        loss.backward()

        # Update weights
        optimizer.step()

        # Track loss and accuracy
        total_loss += loss.item() * images.size(0)

        predictions = outputs.argmax(1)
        correct += (predictions == labels).sum().item()
        total += labels.size(0)

    epoch_loss = total_loss / total
    epoch_acc = correct / total

    return epoch_loss, epoch_acc

In [40]:
best_val_acc = 0

for epoch in range(CFG["epochs"]):

    start = time.time()

    train_loss, train_acc = train_one_epoch(
        model, train_dl, criterion, optimizer, device
    )

    val_loss, val_acc = evaluate(
        model, val_dl, criterion, device
    )

    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save(model.state_dict(), "best.pt")

    elapsed = time.time() - start

    print(
        f"Epoch [{epoch+1}/{CFG['epochs']}] "
        f"Train Loss: {train_loss:.4f} "
        f"Train Acc: {train_acc:.4f} "
        f"Val Loss: {val_loss:.4f} "
        f"Val Acc: {val_acc:.4f} "
        f"Time: {elapsed:.1f}s"
    )

Epoch [1/15] Train Loss: 3.5970 Train Acc: 0.0332 Val Loss: 3.5152 Val Acc: 0.0453 Time: 11.5s
Epoch [2/15] Train Loss: 3.4096 Train Acc: 0.0807 Val Loss: 3.3058 Val Acc: 0.0933 Time: 11.4s
Epoch [3/15] Train Loss: 3.1944 Train Acc: 0.1205 Val Loss: 3.1126 Val Acc: 0.1256 Time: 11.4s
Epoch [4/15] Train Loss: 2.8673 Train Acc: 0.2001 Val Loss: 3.0424 Val Acc: 0.1451 Time: 11.7s
Epoch [5/15] Train Loss: 2.5559 Train Acc: 0.2758 Val Loss: 3.0032 Val Acc: 0.1775 Time: 11.5s
Epoch [6/15] Train Loss: 2.1426 Train Acc: 0.3822 Val Loss: 3.1094 Val Acc: 0.1749 Time: 11.6s
Epoch [7/15] Train Loss: 1.5956 Train Acc: 0.5316 Val Loss: 3.5766 Val Acc: 0.1723 Time: 11.8s
Epoch [8/15] Train Loss: 0.9665 Train Acc: 0.7129 Val Loss: 3.9233 Val Acc: 0.1826 Time: 11.7s
Epoch [9/15] Train Loss: 0.5454 Train Acc: 0.8296 Val Loss: 4.6795 Val Acc: 0.1580 Time: 11.6s
Epoch [10/15] Train Loss: 0.3193 Train Acc: 0.9037 Val Loss: 5.1605 Val Acc: 0.1658 Time: 11.4s
Epoch [11/15] Train Loss: 0.2265 Train Acc: 0.931

In [41]:
torch.save(model.state_dict(), "best_random_split.pt")

## 3. Predict and create `submission.csv` (provided, do not edit)